# Notebook 2 — Base Model: Lightweight Temporal–Spatial CNN

**Base model template notebook** for EEG-based neurological disorder detection.

- Focus disorder (example): **Parkinson’s Disease (PD)**
- Input: preprocessed EEG epochs shaped `(N, C, T)`
- Output: class logits for `PD vs Healthy` (binary)

> This notebook is a **code structure template**. Replace the data-loading and preprocessing placeholders with your dataset pipeline.


In [ ]:
# ===== Imports =====
import os
from dataclasses import dataclass
from pathlib import Path
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix, classification_report

# Optional: EEG preprocessing (use if you load raw EEG here)
# import mne

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("Using device:", DEVICE)

def set_seed(seed: int = 42):
    import random
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_seed(42)


In [ ]:
# ===== Config =====
@dataclass
class Config:
    # Data paths (relative to this notebook in models/ folder)
    data_dir: str = "../data/processed/ds004584"
    
    # Model parameters (will be set dynamically after loading data)
    n_channels: int = 60      # Will be updated based on actual data
    n_classes: int = 2        # Binary: PD (1) vs Control (0)
    sampling_rate: int = 250  # From preprocessing
    epoch_seconds: float = 2.0

    # Training — TUNED for better accuracy
    batch_size: int = 32       # Smaller batches → better generalization
    lr: float = 5e-4           # Reduced from 1e-3 → more stable convergence
    weight_decay: float = 5e-4 # Increased regularization
    epochs: int = 50           # More training time
    patience: int = 10         # More patience before early stop
    
    # Class weighting (address imbalance)
    use_weighted_loss: bool = True
    
    # Data augmentation
    use_augmentation: bool = True
    noise_std: float = 0.05        # Gaussian noise
    time_shift_max: int = 10       # Random circular shift ±10 samples
    scale_range: tuple = (0.9, 1.1) # Random amplitude scaling

cfg = Config()
cfg

## Data loading

Recommended approach:

1. **Preprocess raw EEG** using MNE (filter, notch, ICA, re-reference).
2. **Epoch/segment** into fixed windows (e.g., 2s).
3. Save as `npz` with:
   - `X`: float32 array `(N, C, T)`
   - `y`: int64 array `(N,)` labels (0=HC, 1=PD)
   - optional: `subject_id`: `(N,)` to do subject-wise splits.


In [ ]:
# ===== Load preprocessed data (train/val/test splits) =====
def load_split(data_dir: str, split: str):
    """Load a data split (train/val/test) from .npz file."""
    path = Path(data_dir) / f"{split}.npz"
    if not path.exists():
        raise FileNotFoundError(
            f"Expected preprocessed data at {path}.\n"
            "Run preprocessing/preprocess_ds004584.ipynb first to generate the data."
        )
    data = np.load(path, allow_pickle=True)
    X = data["X"].astype(np.float32)  # (N, C, T)
    y = data["y"].astype(np.int64)    # (N,)
    subject_id = data["subject_id"] if "subject_id" in data.files else None
    return X, y, subject_id

# Load all splits
print("Loading preprocessed data...")
X_train, y_train, sid_train = load_split(cfg.data_dir, "train")
X_val, y_val, sid_val = load_split(cfg.data_dir, "val")
X_test, y_test, sid_test = load_split(cfg.data_dir, "test")

# Update config with actual dimensions
cfg.n_channels = X_train.shape[1]
n_samples = X_train.shape[2]

print(f"\n{'='*50}")
print("DATASET LOADED")
print(f"{'='*50}")
print(f"Train: {X_train.shape} | PD: {(y_train==1).sum()}, Control: {(y_train==0).sum()}")
print(f"Val:   {X_val.shape} | PD: {(y_val==1).sum()}, Control: {(y_val==0).sum()}")
print(f"Test:  {X_test.shape} | PD: {(y_test==1).sum()}, Control: {(y_test==0).sum()}")
print(f"\nChannels: {cfg.n_channels}, Samples per epoch: {n_samples}")

In [ ]:
# ===== Torch Dataset with Augmentation =====
class EEGDataset(Dataset):
    def __init__(self, X, y, augment=False, noise_std=0.05, time_shift_max=10, scale_range=(0.9, 1.1)):
        self.X = torch.from_numpy(X)  # (N,C,T)
        self.y = torch.from_numpy(y)  # (N,)
        self.augment = augment
        self.noise_std = noise_std
        self.time_shift_max = time_shift_max
        self.scale_range = scale_range
        
    def __len__(self):
        return len(self.y)
    
    def __getitem__(self, idx):
        x = self.X[idx].clone()  # (C, T)
        if self.augment:
            # 1. Gaussian noise
            x = x + torch.randn_like(x) * self.noise_std
            # 2. Random amplitude scaling
            scale = self.scale_range[0] + (self.scale_range[1] - self.scale_range[0]) * torch.rand(1)
            x = x * scale
            # 3. Random circular time shift
            shift = torch.randint(-self.time_shift_max, self.time_shift_max + 1, (1,)).item()
            if shift != 0:
                x = torch.roll(x, shifts=shift, dims=-1)
        return x, self.y[idx]

def make_loaders(X_train, y_train, X_val, y_val, batch_size, augment_train=False):
    train_ds = EEGDataset(X_train, y_train, augment=augment_train,
                          noise_std=cfg.noise_std, time_shift_max=cfg.time_shift_max,
                          scale_range=cfg.scale_range)
    val_ds = EEGDataset(X_val, y_val, augment=False)
    train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True, drop_last=False)
    val_loader = DataLoader(val_ds, batch_size=batch_size, shuffle=False, drop_last=False)
    return train_loader, val_loader

## Model: Lightweight Temporal–Spatial CNN

Structure idea:
- **Temporal Conv** (learn rhythms)
- **Spatial Conv** (learn channel interactions)
- Pooling + dropout
- Linear classifier

This is easy to explain and a strong baseline.

In [ ]:
# ===== Lightweight Temporal–Spatial CNN (v2: multi-scale + 64 filters) =====
class MultiScaleTemporal(nn.Module):
    """Multi-scale temporal convolutions to capture different EEG frequency bands."""
    def __init__(self, out_per_scale=8):
        super().__init__()
        # Short kernel → fast oscillations (beta/gamma ~13-100 Hz)
        self.conv_short = nn.Sequential(
            nn.Conv2d(1, out_per_scale, kernel_size=(1, 13), padding=(0, 6), bias=False),
            nn.BatchNorm2d(out_per_scale),
            nn.ELU()
        )
        # Medium kernel → mid-range (alpha/beta ~8-30 Hz)
        self.conv_mid = nn.Sequential(
            nn.Conv2d(1, out_per_scale, kernel_size=(1, 25), padding=(0, 12), bias=False),
            nn.BatchNorm2d(out_per_scale),
            nn.ELU()
        )
        # Long kernel → slow oscillations (delta/theta ~1-8 Hz)
        self.conv_long = nn.Sequential(
            nn.Conv2d(1, out_per_scale, kernel_size=(1, 51), padding=(0, 25), bias=False),
            nn.BatchNorm2d(out_per_scale),
            nn.ELU()
        )
    
    def forward(self, x):
        # x: (B, 1, C, T) → concatenate along channel dim → (B, 3*out_per_scale, C, T)
        return torch.cat([self.conv_short(x), self.conv_mid(x), self.conv_long(x)], dim=1)


class LWTemporalSpatialCNN(nn.Module):
    def __init__(self, n_channels, n_classes, T, dropout=0.5):
        super().__init__()
        temporal_out = 24  # 8 filters × 3 scales
        spatial_out = 64   # Increased from 32 → 64
        
        # Input: (B,C,T) → (B,1,C,T)
        self.temporal = MultiScaleTemporal(out_per_scale=8)
        
        # Spatial conv across channels (increased capacity)
        self.spatial = nn.Sequential(
            nn.Conv2d(temporal_out, spatial_out, kernel_size=(n_channels, 1), bias=False),
            nn.BatchNorm2d(spatial_out),
            nn.ELU()
        )
        self.pool = nn.Sequential(
            nn.AvgPool2d(kernel_size=(1, 4)),
            nn.Dropout(dropout)
        )
        # Refinement block (increased to 64 filters)
        self.refine = nn.Sequential(
            nn.Conv2d(spatial_out, spatial_out, kernel_size=(1, 9), padding=(0, 4), bias=False),
            nn.BatchNorm2d(spatial_out),
            nn.ELU(),
            nn.AvgPool2d(kernel_size=(1, 4)),
            nn.Dropout(dropout)
        )
        # Compute feature dim dynamically
        with torch.no_grad():
            dummy = torch.zeros(1, 1, n_channels, T)
            out = self.refine(self.pool(self.spatial(self.temporal(dummy))))
            feat_dim = out.view(1, -1).shape[1]
        self.classifier = nn.Linear(feat_dim, n_classes)
        self.conf_head = nn.Linear(feat_dim, 1)

    def forward(self, x, return_features=False):
        x = x.unsqueeze(1)       # (B,C,T) → (B,1,C,T)
        x = self.temporal(x)     # Multi-scale temporal features
        x = self.spatial(x)      # Spatial mixing
        x = self.pool(x)
        x = self.refine(x)
        feats = x.flatten(1)
        logits = self.classifier(feats)
        conf = torch.sigmoid(self.conf_head(feats))
        if return_features:
            return logits, conf, feats
        return logits

T = n_samples
model = LWTemporalSpatialCNN(cfg.n_channels, cfg.n_classes, T=T).to(DEVICE)
print(model)
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"\nTotal parameters: {total_params:,}")
print(f"Trainable parameters: {trainable_params:,}")

In [ ]:
# ===== Training & Evaluation utilities =====
def train_one_epoch(model, loader, optimizer, criterion):
    model.train()
    total_loss = 0.0
    for xb, yb in loader:
        xb = xb.to(DEVICE)
        yb = yb.to(DEVICE)
        optimizer.zero_grad()
        logits = model(xb)
        loss = criterion(logits, yb)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    return total_loss / max(1, len(loader))

def eval_model(model, loader):
    model.eval()
    y_true, y_pred = [], []
    with torch.no_grad():
        for xb, yb in loader:
            xb = xb.to(DEVICE)
            logits = model(xb)
            pred = torch.argmax(logits, dim=1).cpu().numpy()
            y_true.append(yb.numpy())
            y_pred.append(pred)
    y_true = np.concatenate(y_true)
    y_pred = np.concatenate(y_pred)
    acc = accuracy_score(y_true, y_pred)
    f1 = f1_score(y_true, y_pred)
    cm = confusion_matrix(y_true, y_pred)
    return acc, f1, cm, y_true, y_pred


## Training with early stopping

Using the existing subject-wise train/val/test splits from preprocessing.
Training with early stopping on validation F1 score.

In [ ]:
# ===== Training with early stopping + mild class weighting + label smoothing =====
train_loader, val_loader = make_loaders(
    X_train, y_train, X_val, y_val, cfg.batch_size, 
    augment_train=cfg.use_augmentation
)
test_ds = EEGDataset(X_test, y_test, augment=False)
test_loader = DataLoader(test_ds, batch_size=cfg.batch_size, shuffle=False)

optimizer = optim.AdamW(model.parameters(), lr=cfg.lr, weight_decay=cfg.weight_decay)

# Mild class weighting (Control=1.2, PD=1.0) + label smoothing
if cfg.use_weighted_loss:
    class_weights = torch.tensor([1.2, 1.0], dtype=torch.float32).to(DEVICE)
    print(f"Class weights: Control={class_weights[0]:.3f}, PD={class_weights[1]:.3f}")
    criterion = nn.CrossEntropyLoss(weight=class_weights, label_smoothing=0.1)
else:
    criterion = nn.CrossEntropyLoss(label_smoothing=0.1)

print(f"Label smoothing: 0.1")

scheduler = optim.lr_scheduler.CosineAnnealingWarmRestarts(optimizer, T_0=10, T_mult=2)

best_f1 = -1
best_state = None
patience_counter = 0
train_losses = []
val_accs = []
val_f1s = []

print(f"\nTraining for up to {cfg.epochs} epochs (patience={cfg.patience})...")
print(f"Augmentation: {cfg.use_augmentation} | Weighted loss: {cfg.use_weighted_loss}\n")

for epoch in range(1, cfg.epochs + 1):
    loss = train_one_epoch(model, train_loader, optimizer, criterion)
    acc, f1, cm, _, _ = eval_model(model, val_loader)
    
    train_losses.append(loss)
    val_accs.append(acc)
    val_f1s.append(f1)
    
    scheduler.step()
    
    if f1 > best_f1:
        best_f1 = f1
        best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
        patience_counter = 0
        marker = " *** best ***"
    else:
        patience_counter += 1
        marker = ""
    
    if epoch % 3 == 0 or epoch == 1 or marker:
        print(f"Epoch {epoch:03d} | loss={loss:.4f} | val_acc={acc:.4f} | val_f1={f1:.4f}{marker}")
    
    if patience_counter >= cfg.patience:
        print(f"\nEarly stopping at epoch {epoch} (no improvement for {cfg.patience} epochs)")
        break

# Load best model
model.load_state_dict(best_state)
print(f"\nBest validation F1: {best_f1:.4f}")

## Test Set Evaluation & Visualization

In [ ]:
# ===== Final Test Evaluation =====
import matplotlib.pyplot as plt
from sklearn.metrics import ConfusionMatrixDisplay, roc_curve, auc, precision_recall_curve

acc_test, f1_test, cm_test, y_true_test, y_pred_test = eval_model(model, test_loader)

print("=" * 50)
print("TEST SET RESULTS")
print("=" * 50)
print(f"Accuracy:  {acc_test:.4f}")
print(f"F1 Score:  {f1_test:.4f}")
print(f"\nConfusion Matrix:\n{cm_test}")
print(f"\n{classification_report(y_true_test, y_pred_test, target_names=['Control', 'PD'], digits=4)}")

# --- Plots ---
fig, axes = plt.subplots(1, 3, figsize=(16, 4))

# 1. Training curves
axes[0].plot(train_losses, label='Train Loss')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Loss')
axes[0].set_title('Training Loss')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# 2. Validation metrics
ax2 = axes[1]
ax2.plot(val_accs, label='Val Accuracy', color='blue')
ax2.plot(val_f1s, label='Val F1', color='orange')
ax2.set_xlabel('Epoch')
ax2.set_ylabel('Score')
ax2.set_title('Validation Metrics')
ax2.legend()
ax2.grid(True, alpha=0.3)

# 3. Confusion matrix
ConfusionMatrixDisplay(cm_test, display_labels=['Control', 'PD']).plot(ax=axes[2], cmap='Blues')
axes[2].set_title('Test Confusion Matrix')

plt.tight_layout()
plt.show()

# Save model
save_dir = Path("../training/saved_models")
save_dir.mkdir(parents=True, exist_ok=True)
save_path = save_dir / "lw_temporal_spatial_cnn_parkinsons.pth"
torch.save(model.state_dict(), save_path)
print(f"\nModel saved to: {save_path}")